## 11 - Auditoria externa ASTA antes da inferência (v3)
Este notebook audita o conjunto real ASTA sem executar YOLO, Hough ou qualquer outro
detector. A auditoria confirma a origem, o inventário exato, os pares imagem–máscara,
dimensões, hashes, legibilidade, valores das máscaras, componentes conectados e casos
que precisam de revisão visual antes de converter máscaras em objetos geométricos.
Regra de proteção: data/raw/asta_real é somente leitura. Checkpoints e relatórios
são gravados exclusivamente em experiments/asta_external_v3/audit. O protocolo de
tiles, agrupamento de componentes, matching e métricas será congelado somente depois
desta auditoria; portanto, este notebook não abre o teste externo.
Execute em CPU. A etapa pesada percorre cerca de 18 GB, mostra progresso e é
retomável por checkpoint. Não copie a pasta inteira para /content: apenas um par é
temporariamente preparado no disco local por vez.

## 0. Ambiente, Drive e caminhos protegidos
Esta seção monta o Drive e declara utilitários. Nenhum arquivo bruto é alterado.

In [1]:
!pip install -q remotezip

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
from datetime import datetime, timezone
from remotezip import RemoteZip
from tqdm.auto import tqdm
from PIL import Image, ImageDraw
import cv2, hashlib, json, math, os, shutil, time, warnings, zipfile
import numpy as np
import pandas as pd
import requests

# 10.560² = 111.513.600 pixels. Mantemos um teto explícito em vez de desativar
# globalmente a proteção contra imagens desproporcionalmente grandes.
Image.MAX_IMAGE_PIXELS = 200_000_000

BACKUP_DIR=Path('/content/drive/MyDrive/tcc-satellite-streaks')
ASTA_RAW=BACKUP_DIR/'data/raw/asta_real'
AUDIT_DRIVE=BACKUP_DIR/'experiments/asta_external_v3/audit'
AUDIT_LOCAL=Path('/content/asta_audit_v3')
STAGING=AUDIT_LOCAL/'staging_um_par'
PREVIEWS_DRIVE=AUDIT_DRIVE/'previews_checkpoint'
RECORD_ID='11642424'
DOI='10.5281/zenodo.11642424'
URL_ZIP='https://zenodo.org/records/11642424/files/Processed.zip?download=1'
URL_API='https://zenodo.org/api/records/11642424'
EXPECTED_ZIP_MD5='37c6c2096c74ce22778d3baad056d0b9'
AREA_MINIMA=15
CHECKPOINT_EVERY=5

assert ASTA_RAW.is_dir(),f'Pasta ASTA ausente no Drive: {ASTA_RAW}'
AUDIT_DRIVE.mkdir(parents=True,exist_ok=True)
AUDIT_LOCAL.mkdir(parents=True,exist_ok=True)
STAGING.mkdir(parents=True,exist_ok=True)
PREVIEWS_DRIVE.mkdir(parents=True,exist_ok=True)

def agora():return datetime.now().strftime('%H:%M:%S')
def sha256_arquivo(caminho,chunk=8*1024*1024):
    h=hashlib.sha256()
    with open(caminho,'rb') as arq:
        for bloco in iter(lambda:arq.read(chunk),b''):h.update(bloco)
    return h.hexdigest()
def sha256_canonico(objeto):
    texto=json.dumps(objeto,sort_keys=True,separators=(',',':'),ensure_ascii=False)
    return hashlib.sha256(texto.encode('utf-8')).hexdigest()
def gravar_json_atomico(caminho,objeto):
    caminho=Path(caminho);caminho.parent.mkdir(parents=True,exist_ok=True)
    temporario=caminho.with_name(caminho.name+'.tmp')
    temporario.write_text(json.dumps(objeto,indent=2,ensure_ascii=False),encoding='utf-8')
    os.replace(temporario,caminho)
def gravar_csv_atomico(caminho,tabela):
    caminho=Path(caminho);caminho.parent.mkdir(parents=True,exist_ok=True)
    temporario=caminho.with_name(caminho.name+'.tmp')
    tabela.to_csv(temporario,index=False)
    os.replace(temporario,caminho)

print(f'[{agora()}] ASTA localizado. Runtime CPU é suficiente.')
print('Fonte bruta protegida (somente leitura):',ASTA_RAW)
print('Saídas da auditoria:',AUDIT_DRIVE)

Mounted at /content/drive
[09:45:46] ASTA localizado. Runtime CPU é suficiente.
Fonte bruta protegida (somente leitura): /content/drive/MyDrive/tcc-satellite-streaks/data/raw/asta_real
Saídas da auditoria: /content/drive/MyDrive/tcc-satellite-streaks/experiments/asta_external_v3/audit


## 1. Inventário oficial e gate leve
O índice do Processed.zip é consultado por requisições parciais; o arquivo de 18 GB
não é baixado. O gate exige os 178 pares oficiais, compara os nomes com o Drive e valida
o registro Zenodo. Ainda não decodifica as imagens nem calcula seus SHA-256.

In [2]:
def nome_imagem_para_mascara(nome):
    sufixo='.fits_full.png'
    if not nome.endswith(sufixo):raise ValueError(f'Nome de imagem inesperado: {nome}')
    return nome[:-len(sufixo)]+'_mask.png'

print(f'[{agora()}] Lendo índice oficial do Zenodo...')
with RemoteZip(URL_ZIP) as zf:
    membros_oficiais=zf.namelist()
imagens_oficiais=sorted(Path(n).name for n in membros_oficiais if n.endswith('_full.png'))
mascaras_oficiais=sorted(Path(n).name for n in membros_oficiais if n.endswith('_mask.png'))
set_imagens_oficiais,set_mascaras_oficiais=set(imagens_oficiais),set(mascaras_oficiais)
pares_oficiais=[]
for nome_img in imagens_oficiais:
    nome_mask=nome_imagem_para_mascara(nome_img)
    if nome_mask in set_mascaras_oficiais:pares_oficiais.append((nome_img,nome_mask))

resposta=requests.get(URL_API,timeout=60);resposta.raise_for_status();zenodo=resposta.json()
arquivo_zip=next(f for f in zenodo['files'] if f['key']=='Processed.zip')
checksum_zip=arquivo_zip['checksum'].split(':',1)[-1].lower()
licenca=(zenodo.get('metadata',{}).get('license') or {}).get('id')

nomes_locais={p.name for p in ASTA_RAW.iterdir() if p.is_file()}
imagens_locais={n for n in nomes_locais if n.endswith('_full.png')}
mascaras_locais={n for n in nomes_locais if n.endswith('_mask.png')}
faltantes_imagem=sorted(set_imagens_oficiais-imagens_locais)
faltantes_mascara=sorted(set_mascaras_oficiais-mascaras_locais)
extras_imagem=sorted(imagens_locais-set_imagens_oficiais)
extras_mascara=sorted(mascaras_locais-set_mascaras_oficiais)

erros=[];avisos=[]
if len(imagens_oficiais)!=178 or len(mascaras_oficiais)!=178 or len(pares_oficiais)!=178:
    erros.append('O índice oficial não contém exatamente 178 pares.')
if checksum_zip!=EXPECTED_ZIP_MD5:erros.append('MD5 oficial do Processed.zip divergiu do registro esperado.')
if faltantes_imagem:erros.append(f'{len(faltantes_imagem)} imagens oficiais ausentes no Drive.')
if faltantes_mascara:erros.append(f'{len(faltantes_mascara)} máscaras oficiais ausentes no Drive.')
if extras_imagem:erros.append(f'{len(extras_imagem)} imagens *_full.png não pertencem ao índice oficial.')
if extras_mascara:erros.append(f'{len(extras_mascara)} máscaras *_mask.png não pertencem ao índice oficial.')
if licenca and licenca.lower()!='cc-by-4.0':avisos.append(f'Licença informada pela API: {licenca!r}; revisar atribuição.')

amostra_path=ASTA_RAW/'amostra_utilizada.csv'
if amostra_path.is_file():
    amostra=pd.read_csv(amostra_path)
    colunas={'arquivo_imagem','arquivo_mascara'}
    if not colunas.issubset(amostra.columns):
        avisos.append('amostra_utilizada.csv não possui as duas colunas históricas esperadas.')
    else:
        pares_amostra=set(map(tuple,amostra[['arquivo_imagem','arquivo_mascara']].astype(str).to_numpy()))
        if pares_amostra!=set(pares_oficiais):avisos.append('amostra_utilizada.csv não coincide exatamente com o índice oficial atual.')
else:avisos.append('amostra_utilizada.csv histórico não foi encontrado; o manifest v3 será a fonte rastreável.')

inventario_tamanho=[
    {'imagem':img,'mascara':mask,'bytes_imagem':(ASTA_RAW/img).stat().st_size,
     'bytes_mascara':(ASTA_RAW/mask).stat().st_size,
     'mtime_ns_imagem':(ASTA_RAW/img).stat().st_mtime_ns,
     'mtime_ns_mascara':(ASTA_RAW/mask).stat().st_mtime_ns}
    for img,mask in pares_oficiais
]
fingerprint_inventario=sha256_canonico(inventario_tamanho)
contrato_inventario={
    'protocolo':'asta_inventory_audit_v3','record_id':RECORD_ID,'doi':DOI,
    'url_zip':URL_ZIP,'zip_md5_oficial':checksum_zip,
    'zip_tamanho_bytes_oficial':int(arquivo_zip['size']),'licenca_api':licenca,
    'pares_oficiais':len(pares_oficiais),'imagens_drive':len(imagens_locais),
    'mascaras_drive':len(mascaras_locais),'faltantes_imagem':faltantes_imagem,
    'faltantes_mascara':faltantes_mascara,'extras_imagem':extras_imagem,
    'extras_mascara':extras_mascara,'fingerprint_nomes_tamanhos':fingerprint_inventario,
    'raw_somente_leitura':True,'yolo_executado':False,'hough_executado':False,
    'erros':erros,'avisos':avisos
}
contrato_inventario['sha256_contrato_inventario']=sha256_canonico(contrato_inventario)
print(json.dumps(contrato_inventario,indent=2,ensure_ascii=False))
if erros:raise RuntimeError('Gate de inventário ASTA reprovado; não execute a auditoria pesada.')
print('INVENTARIO_ASTA_APROVADO_PARA_AUDITORIA_PESADA')
print('Compartilhe este relatório antes de habilitar a seção 2.')

[09:45:48] Lendo índice oficial do Zenodo...
{
  "protocolo": "asta_inventory_audit_v3",
  "record_id": "11642424",
  "doi": "10.5281/zenodo.11642424",
  "url_zip": "https://zenodo.org/records/11642424/files/Processed.zip?download=1",
  "zip_md5_oficial": "37c6c2096c74ce22778d3baad056d0b9",
  "zip_tamanho_bytes_oficial": 17950002539,
  "licenca_api": "cc-by-4.0",
  "pares_oficiais": 178,
  "imagens_drive": 178,
  "mascaras_drive": 178,
  "faltantes_imagem": [],
  "faltantes_mascara": [],
  "extras_imagem": [],
  "extras_mascara": [],
  "fingerprint_nomes_tamanhos": "6b307e94a0894518016da27c5245b729d113471d6af42393bd6fc167941fb6e0",
  "raw_somente_leitura": true,
  "yolo_executado": false,
  "hough_executado": false,
  "erros": [],
  "avisos": [],
  "sha256_contrato_inventario": "ca4609ef651bd30cef611bd394c539868b10f2e8547676e4d00af7e271c14c06"
}
INVENTARIO_ASTA_APROVADO_PARA_AUDITORIA_PESADA
Compartilhe este relatório antes de habilitar a seção 2.


## 2. Auditoria pesada, retomável e sem inferência
Primeiro execute com EXECUTAR_AUDITORIA_PESADA=False. Após conferir o gate leve,
altere somente essa chave para True. Cada par é copiado para uma pasta temporária,
auditado e removido; checkpoints são publicados a cada cinco pares. Uma queda da sessão
não obriga a repetir os pares já confirmados sob o mesmo fingerprint de inventário.
As folhas mostram uma miniatura da imagem com a máscara em vermelho. Componentes
conectados são sinais para revisão, não equivalem automaticamente ao número de
trilhas: uma trilha pode ser interrompida e uma imagem pode conter várias trilhas.

In [3]:
EXECUTAR_AUDITORIA_PESADA=True

CHECKPOINT_MANIFEST=AUDIT_DRIVE/'checkpoint_manifest_auditoria_asta_v3.csv'
CHECKPOINT_COMPONENTES=AUDIT_DRIVE/'checkpoint_componentes_mascaras_asta_v3.csv'
CHECKPOINT_CONTRATO=AUDIT_DRIVE/'checkpoint_contrato_auditoria_asta_v3.json'

def copiar_para_stage_com_hash(origem,destino,tentativas=3,chunk=8*1024*1024):
    ultimo=None
    for tentativa in range(1,tentativas+1):
        try:
            if destino.exists():destino.unlink()
            h=hashlib.sha256()
            with origem.open('rb') as entrada,destino.open('wb') as saida:
                for bloco in iter(lambda:entrada.read(chunk),b''):
                    saida.write(bloco);h.update(bloco)
            if destino.stat().st_size!=origem.stat().st_size:raise IOError('tamanho local divergente após cópia')
            return h.hexdigest()
        except (OSError,IOError) as exc:
            ultimo=exc
            if destino.exists():destino.unlink()
            if tentativa<tentativas:time.sleep(2**tentativa)
    raise IOError(f'Falha ao preparar {origem.name} após {tentativas} tentativas: {ultimo}')

def orientacao_componente(labels,indice,stat):
    x,y,w,h,area=[int(v) for v in stat]
    recorte=labels[y:y+h,x:x+w]
    yy,xx=np.where(recorte==indice)
    if len(xx)>200_000:
        passo=math.ceil(len(xx)/200_000);xx=xx[::passo];yy=yy[::passo]
    if len(xx)<2:return np.nan
    pts=np.column_stack([xx.astype(np.float32)+x,yy.astype(np.float32)+y]).reshape(-1,1,2)
    vx,vy,_,_=cv2.fitLine(pts,cv2.DIST_L2,0,0.01,0.01).reshape(-1)
    return float(np.degrees(np.arctan2(vy,vx))%180.0)

def salvar_preview(caminho_img,mask,nome_saida,titulo):
    with warnings.catch_warnings():
        warnings.simplefilter('error',Image.DecompressionBombWarning)
        with Image.open(caminho_img) as pil:
            modo=pil.mode
            cinza=pil.convert('L');cinza.thumbnail((512,512),Image.Resampling.LANCZOS)
            base=np.asarray(cinza,dtype=np.uint8)
    mini_mask=cv2.resize((mask>0).astype(np.uint8),(base.shape[1],base.shape[0]),interpolation=cv2.INTER_NEAREST)
    rgb=np.repeat(base[:,:,None],3,axis=2);ativo=mini_mask.astype(bool)
    rgb[ativo,0]=255;rgb[ativo,1]=(rgb[ativo,1]*0.25).astype(np.uint8);rgb[ativo,2]=(rgb[ativo,2]*0.25).astype(np.uint8)
    quadro=Image.new('RGB',(512,548),'white');quadro.paste(Image.fromarray(rgb),(0,36))
    ImageDraw.Draw(quadro).text((8,10),titulo[:74],fill='black')
    temporario=AUDIT_LOCAL/(nome_saida+'.tmp.jpg');quadro.save(temporario,quality=88,optimize=True)
    destino=PREVIEWS_DRIVE/nome_saida;shutil.copy2(temporario,destino);temporario.unlink()
    return modo,base

def auditar_par(nome_img,nome_mask):
    origem_img,origem_mask=ASTA_RAW/nome_img,ASTA_RAW/nome_mask
    local_img,local_mask=STAGING/nome_img,STAGING/nome_mask
    hash_img=copiar_para_stage_com_hash(origem_img,local_img)
    hash_mask=copiar_para_stage_com_hash(origem_mask,local_mask)
    try:
        with Image.open(local_img) as pil_img:
            largura_img,altura_img=pil_img.size;modo_img=pil_img.mode;formato_img=pil_img.format
        with Image.open(local_mask) as pil_mask:
            largura_mask,altura_mask=pil_mask.size;modo_mask=pil_mask.mode;formato_mask=pil_mask.format
        mask_original=cv2.imread(str(local_mask),cv2.IMREAD_UNCHANGED)
        if mask_original is None:raise IOError('cv2 não conseguiu ler a máscara')
        canais_mask=1 if mask_original.ndim==2 else int(mask_original.shape[2])
        dtype_mask=str(mask_original.dtype)
        if mask_original.ndim==2:mask=mask_original
        elif mask_original.shape[2]==4:mask=cv2.cvtColor(mask_original,cv2.COLOR_BGRA2GRAY)
        else:mask=cv2.cvtColor(mask_original,cv2.COLOR_BGR2GRAY)
        if mask.shape!=(altura_mask,largura_mask):raise ValueError('shape da máscara diverge do cabeçalho PIL')
        if mask.dtype==np.uint8:
            hist=cv2.calcHist([mask],[0],None,[256],[0,256]).ravel();valores_mask=[int(i) for i,v in enumerate(hist) if v>0]
        else:valores_mask=[int(x) for x in np.unique(mask)]
        binaria=np.where(mask>0,255,0).astype(np.uint8)
        n_labels,labels,stats,centroids=cv2.connectedComponentsWithStats(binaria,connectivity=8)
        componentes=[]
        for indice in range(1,n_labels):
            x,y,w,h,area=[int(v) for v in stats[indice]]
            if area<AREA_MINIMA:continue
            componentes.append({
                'id_asta':Path(nome_img).name.replace('.fits_full.png',''),'arquivo_mascara':nome_mask,
                'componente_indice':indice,'area_pixels':area,'x':x,'y':y,'largura_bbox':w,'altura_bbox':h,
                'centro_x':float(centroids[indice][0]),'centro_y':float(centroids[indice][1]),
                'angulo_graus_0_180':orientacao_componente(labels,indice,stats[indice]),
                'toca_borda':bool(x==0 or y==0 or x+w==largura_mask or y+h==altura_mask)
            })
        id_asta=Path(nome_img).name.replace('.fits_full.png','')
        preview_nome=id_asta+'.jpg'
        _,mini=salvar_preview(local_img,mask,preview_nome,f'{id_asta} | componentes >= {AREA_MINIMA}: {len(componentes)}')
        p=np.percentile(mini,[1,50,99,99.8]).astype(float)
        linha={
            'id_asta':id_asta,'arquivo_imagem':nome_img,'arquivo_mascara':nome_mask,
            'bytes_imagem':origem_img.stat().st_size,'bytes_mascara':origem_mask.stat().st_size,
            'sha256_imagem':hash_img,'sha256_mascara':hash_mask,
            'largura_imagem':largura_img,'altura_imagem':altura_img,'modo_imagem':modo_img,'formato_imagem':formato_img,
            'largura_mascara':largura_mask,'altura_mascara':altura_mask,'modo_mascara':modo_mask,'formato_mascara':formato_mask,
            'dtype_mascara':dtype_mask,'canais_mascara':canais_mask,
            'dimensoes_coincidem':bool((largura_img,altura_img)==(largura_mask,altura_mask)),
            'valores_mascara':'|'.join(map(str,valores_mask)),'mascara_pixels_positivos':int(cv2.countNonZero(binaria)),
            'mascara_fracao_positiva':float(cv2.countNonZero(binaria)/binaria.size),
            'componentes_totais':int(n_labels-1),'componentes_relevantes':len(componentes),
            'componentes_pequenos':int((n_labels-1)-len(componentes)),
            'componentes_relevantes_tocam_borda':int(sum(c['toca_borda'] for c in componentes)),
            'imagem_p1_miniatura':p[0],'imagem_p50_miniatura':p[1],
            'imagem_p99_miniatura':p[2],'imagem_p998_miniatura':p[3],
            'imagem_desvio_miniatura':float(mini.std()),'preview_relativo':str(preview_nome)
        }
        del labels,stats,centroids,binaria,mask,mask_original,mini
        return linha,componentes
    finally:
        if local_img.exists():local_img.unlink()
        if local_mask.exists():local_mask.unlink()

def publicar_checkpoint(linhas,componentes):
    gravar_csv_atomico(CHECKPOINT_MANIFEST,pd.DataFrame(linhas).sort_values('id_asta'))
    comp_df=pd.DataFrame(componentes)
    if len(comp_df):comp_df=comp_df.sort_values(['id_asta','componente_indice'])
    gravar_csv_atomico(CHECKPOINT_COMPONENTES,comp_df)
    gravar_json_atomico(CHECKPOINT_CONTRATO,{
        'protocolo':'asta_audit_checkpoint_v3','fingerprint_nomes_tamanhos':fingerprint_inventario,
        'pares_concluidos':len(linhas),'atualizado_em_utc':datetime.now(timezone.utc).isoformat(),
        'raw_somente_leitura':True,'yolo_executado':False,'hough_executado':False
    })

if not EXECUTAR_AUDITORIA_PESADA:
    print('Auditoria pesada bloqueada. Compartilhe primeiro o gate leve da seção 1.')
else:
    contrato_checkpoint=json.loads(CHECKPOINT_CONTRATO.read_text(encoding='utf-8')) if CHECKPOINT_CONTRATO.exists() else None
    if contrato_checkpoint:
        assert contrato_checkpoint['fingerprint_nomes_tamanhos']==fingerprint_inventario,'Inventário mudou; checkpoint não pode ser reutilizado.'
    linhas_existentes=pd.read_csv(CHECKPOINT_MANIFEST).to_dict('records') if CHECKPOINT_MANIFEST.exists() else []
    if CHECKPOINT_COMPONENTES.exists() and CHECKPOINT_COMPONENTES.stat().st_size:
        try:componentes_existentes=pd.read_csv(CHECKPOINT_COMPONENTES).to_dict('records')
        except pd.errors.EmptyDataError:componentes_existentes=[]
    else:componentes_existentes=[]
    por_id={str(x['id_asta']):x for x in linhas_existentes}
    componentes_por_id={}
    for c in componentes_existentes:componentes_por_id.setdefault(str(c['id_asta']),[]).append(c)
    bytes_total=sum(x['bytes_imagem']+x['bytes_mascara'] for x in inventario_tamanho)
    bytes_novos=0;novos_desde_checkpoint=0;inicio=time.time()
    barra=tqdm(pares_oficiais,total=len(pares_oficiais),desc='Auditoria ASTA',unit='par')
    for nome_img,nome_mask in barra:
        id_asta=nome_img.replace('.fits_full.png','')
        preview_ok=(PREVIEWS_DRIVE/(id_asta+'.jpg')).is_file()
        if id_asta in por_id and preview_ok:
            barra.set_postfix(concluidos=len(por_id),retomados='sim');continue
        linha,componentes=auditar_par(nome_img,nome_mask)
        por_id[id_asta]=linha;componentes_por_id[id_asta]=componentes
        bytes_novos+=linha['bytes_imagem']+linha['bytes_mascara'];novos_desde_checkpoint+=1
        decorrido=max(time.time()-inicio,1e-9);taxa=bytes_novos/decorrido
        restante=max(0,bytes_total-bytes_novos)
        barra.set_postfix(concluidos=len(por_id),GB=f'{bytes_novos/1e9:.2f}',MBps=f'{taxa/1e6:.1f}',eta_min=f'{restante/max(taxa,1)/60:.1f}')
        if novos_desde_checkpoint>=CHECKPOINT_EVERY:
            todas_linhas=list(por_id.values());todos_componentes=[c for grupo in componentes_por_id.values() for c in grupo]
            publicar_checkpoint(todas_linhas,todos_componentes);novos_desde_checkpoint=0
            print(f'[{agora()}] Checkpoint publicado: {len(todas_linhas)}/{len(pares_oficiais)} pares.')
    linhas_auditadas=list(por_id.values());componentes_auditados=[c for grupo in componentes_por_id.values() for c in grupo]
    publicar_checkpoint(linhas_auditadas,componentes_auditados)
    print(f'[{agora()}] Auditoria automática concluída: {len(linhas_auditadas)} pares.')

Auditoria ASTA:   0%|          | 0/178 [00:00<?, ?par/s]

[09:46:03] Auditoria automática concluída: 178 pares.


## 3. Consolidação, folhas visuais e backup verificável
Esta seção também não executa modelos. Ela exige os 178 pares auditados, produz folhas
de revisão com todas as imagens, registra alertas objetivos e publica artefatos finais
com SHA-256. O resultado automático não aprova ainda a conversão das máscaras.

In [4]:
if not CHECKPOINT_MANIFEST.exists():
    print('Auditoria pesada ainda não executada.')
else:
    manifest_asta=pd.read_csv(CHECKPOINT_MANIFEST).sort_values('id_asta').reset_index(drop=True)
    if CHECKPOINT_COMPONENTES.exists() and CHECKPOINT_COMPONENTES.stat().st_size:
        try:componentes_asta=pd.read_csv(CHECKPOINT_COMPONENTES)
        except pd.errors.EmptyDataError:componentes_asta=pd.DataFrame()
    else:componentes_asta=pd.DataFrame()
    erros=[];avisos=[]
    inventario_atual=[
        {'imagem':img,'mascara':mask,'bytes_imagem':(ASTA_RAW/img).stat().st_size,
         'bytes_mascara':(ASTA_RAW/mask).stat().st_size,
         'mtime_ns_imagem':(ASTA_RAW/img).stat().st_mtime_ns,
         'mtime_ns_mascara':(ASTA_RAW/mask).stat().st_mtime_ns}
        for img,mask in pares_oficiais
    ]
    if sha256_canonico(inventario_atual)!=fingerprint_inventario:erros.append('Inventário raw mudou durante a auditoria.')
    if len(manifest_asta)!=178:erros.append(f'Esperados 178 pares auditados; encontrados {len(manifest_asta)}.')
    if manifest_asta.id_asta.nunique()!=len(manifest_asta):erros.append('IDs ASTA duplicados no manifest.')
    if not manifest_asta.dimensoes_coincidem.astype(bool).all():erros.append('Há pares com dimensões imagem/máscara divergentes.')
    if manifest_asta.sha256_imagem.duplicated().any():avisos.append('Há imagens com SHA-256 duplicado; revisar proveniência.')
    if manifest_asta.sha256_mascara.duplicated().any():avisos.append('Há máscaras com SHA-256 duplicado; revisar conteúdo.')
    vazias=int((manifest_asta.mascara_pixels_positivos==0).sum())
    multicomponentes=int((manifest_asta.componentes_relevantes>1).sum())
    sem_componente_relevante=int((manifest_asta.componentes_relevantes==0).sum())
    toca_borda=int((manifest_asta.componentes_relevantes_tocam_borda>0).sum())
    if vazias:avisos.append(f'{vazias} máscaras vazias exigem revisão.')
    if sem_componente_relevante:avisos.append(f'{sem_componente_relevante} máscaras sem componente >= {AREA_MINIMA} exigem revisão.')
    if multicomponentes:avisos.append(f'{multicomponentes} máscaras possuem múltiplos componentes relevantes; isso não define sozinho o número de trilhas.')

    FOLHAS_LOCAL=AUDIT_LOCAL/'folhas_auditoria_asta_v3';FOLHAS_LOCAL.mkdir(parents=True,exist_ok=True)
    previews=[]
    for linha in manifest_asta.itertuples():
        caminho=PREVIEWS_DRIVE/linha.preview_relativo
        if not caminho.is_file():erros.append(f'Preview ausente: {caminho.name}')
        else:previews.append((linha.id_asta,caminho,int(linha.componentes_relevantes)))
    for antigo in FOLHAS_LOCAL.glob('folha_*.jpg'):antigo.unlink()
    por_folha=12;colunas=4;largura_item,altura_item=384,411
    for inicio_indice in range(0,len(previews),por_folha):
        lote=previews[inicio_indice:inicio_indice+por_folha];linhas_grade=math.ceil(len(lote)/colunas)
        folha=Image.new('RGB',(colunas*largura_item,linhas_grade*altura_item),(235,235,235))
        for pos,(id_asta,caminho,n_comp) in enumerate(lote):
            with Image.open(caminho) as prev:
                prev=prev.convert('RGB');prev.thumbnail((largura_item,altura_item-24),Image.Resampling.LANCZOS)
                x=(pos%colunas)*largura_item+(largura_item-prev.width)//2
                y=(pos//colunas)*altura_item+24;folha.paste(prev,(x,y))
            ImageDraw.Draw(folha).text(((pos%colunas)*largura_item+5,(pos//colunas)*altura_item+5),f'{id_asta} | comp={n_comp}',fill='black')
        folha.save(FOLHAS_LOCAL/f'folha_{inicio_indice//por_folha+1:02d}.jpg',quality=88,optimize=True)

    FINAL_LOCAL=AUDIT_LOCAL/'final';FINAL_LOCAL.mkdir(parents=True,exist_ok=True)
    manifest_path=FINAL_LOCAL/'manifest_auditoria_asta_v3.csv';manifest_asta.to_csv(manifest_path,index=False)
    componentes_path=FINAL_LOCAL/'componentes_mascaras_asta_v3.csv';componentes_asta.to_csv(componentes_path,index=False)
    contrato_path=FINAL_LOCAL/'contrato_inventario_asta_v3.json';contrato_path.write_text(json.dumps(contrato_inventario,indent=2,ensure_ascii=False),encoding='utf-8')
    zip_path=FINAL_LOCAL/'folhas_auditoria_asta_v3.zip'
    with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as zf:
        for folha in sorted(FOLHAS_LOCAL.glob('folha_*.jpg')):zf.write(folha,arcname=folha.name)

    dimensoes=(manifest_asta.groupby(['largura_imagem','altura_imagem']).size().reset_index(name='quantidade').to_dict('records'))
    modos=manifest_asta.modo_imagem.value_counts().to_dict();valores=manifest_asta.valores_mascara.value_counts().to_dict()
    resumo={
        'protocolo':'asta_external_audit_v3','status':'AUDITORIA_AUTOMATICA_ASTA_CONCLUIDA_AGUARDA_REVISAO_VISUAL',
        'record_id':RECORD_ID,'doi':DOI,'pares_auditados':len(manifest_asta),
        'fingerprint_nomes_tamanhos':fingerprint_inventario,
        'sha256_contrato_inventario':contrato_inventario['sha256_contrato_inventario'],
        'dimensoes_imagem':dimensoes,'modos_imagem':modos,'valores_mascara':valores,
        'mascaras_vazias':vazias,'sem_componente_relevante':sem_componente_relevante,
        'multiplos_componentes_relevantes':multicomponentes,'componentes_tocando_borda':toca_borda,
        'area_minima_componente':AREA_MINIMA,'erros':erros,'avisos':avisos,
        'raw_somente_leitura':True,'yolo_executado':False,'hough_executado':False,
        'protocolo_tiles_congelado':False,'conversao_mascara_objetos_congelada':False,
        'revisao_visual_aprovada':False
    }
    resumo['sha256_manifest_auditoria']=sha256_arquivo(manifest_path)
    resumo['sha256_componentes']=sha256_arquivo(componentes_path)
    resumo['sha256_folhas_zip']=sha256_arquivo(zip_path)
    resumo_path=FINAL_LOCAL/'resumo_auditoria_asta_v3.json';resumo_path.write_text(json.dumps(resumo,indent=2,ensure_ascii=False),encoding='utf-8')
    registro_hashes={p.name:sha256_arquivo(p) for p in sorted(FINAL_LOCAL.iterdir()) if p.is_file()}
    hashes_path=FINAL_LOCAL/'hashes_artefatos_auditoria_asta_v3.json';hashes_path.write_text(json.dumps(registro_hashes,indent=2),encoding='utf-8')
    if erros:
        print(json.dumps(resumo,indent=2,ensure_ascii=False));raise RuntimeError('Consolidação ASTA reprovada; não avançar.')
    FINAL_DRIVE=AUDIT_DRIVE/'final';FINAL_DRIVE.mkdir(parents=True,exist_ok=True)
    for arquivo in tqdm(sorted(FINAL_LOCAL.iterdir()),desc='Publicando auditoria ASTA',unit='arq'):
        if arquivo.is_file():shutil.copy2(arquivo,FINAL_DRIVE/arquivo.name)
    for nome,hash_esperado in registro_hashes.items():assert sha256_arquivo(FINAL_DRIVE/nome)==hash_esperado
    assert sha256_arquivo(FINAL_DRIVE/'hashes_artefatos_auditoria_asta_v3.json')==sha256_arquivo(hashes_path)
    print(json.dumps(resumo,indent=2,ensure_ascii=False))
    print('BACKUP_AUDITORIA_ASTA_AUTOMATICA_APROVADO')
    print('Envie o relatório acima e folhas_auditoria_asta_v3.zip. Nenhuma inferência foi executada.')

Publicando auditoria ASTA:   0%|          | 0/6 [00:00<?, ?arq/s]

{
  "protocolo": "asta_external_audit_v3",
  "status": "AUDITORIA_AUTOMATICA_ASTA_CONCLUIDA_AGUARDA_REVISAO_VISUAL",
  "record_id": "11642424",
  "doi": "10.5281/zenodo.11642424",
  "pares_auditados": 178,
  "fingerprint_nomes_tamanhos": "6b307e94a0894518016da27c5245b729d113471d6af42393bd6fc167941fb6e0",
  "sha256_contrato_inventario": "ca4609ef651bd30cef611bd394c539868b10f2e8547676e4d00af7e271c14c06",
  "dimensoes_imagem": [
    {
      "largura_imagem": 10560,
      "altura_imagem": 10560,
      "quantidade": 178
    }
  ],
  "modos_imagem": {
    "L": 178
  },
  "valores_mascara": {
    "0|255": 177,
    "0": 1
  },
  "mascaras_vazias": 1,
  "sem_componente_relevante": 1,
  "multiplos_componentes_relevantes": 73,
  "componentes_tocando_borda": 0,
  "area_minima_componente": 15,
  "erros": [],
  "avisos": [
    "1 máscaras vazias exigem revisão.",
    "1 máscaras sem componente >= 15 exigem revisão.",
    "73 máscaras possuem múltiplos componentes relevantes; isso não define sozinho 

## 4. Verificação pós-backup, somente leitura
Pode ser executada após reconexão. Confirma os hashes dos artefatos finais sem reler os
18 GB e sem executar detectores. A revisão visual e o protocolo externo continuam
pendentes mesmo quando este gate passa.

In [5]:
FINAL_DRIVE=AUDIT_DRIVE/'final'
hashes_remoto=FINAL_DRIVE/'hashes_artefatos_auditoria_asta_v3.json'
resumo_remoto=FINAL_DRIVE/'resumo_auditoria_asta_v3.json'
if not hashes_remoto.is_file() or not resumo_remoto.is_file():
    print('Backup final da auditoria ainda não existe.')
else:
    hashes=json.loads(hashes_remoto.read_text(encoding='utf-8'))
    problemas=[]
    for nome,esperado in tqdm(sorted(hashes.items()),desc='Verificando artefatos ASTA',unit='arq'):
        caminho=FINAL_DRIVE/nome
        if not caminho.is_file() or sha256_arquivo(caminho)!=esperado:problemas.append(nome)
    resumo_final=json.loads(resumo_remoto.read_text(encoding='utf-8'))
    assert resumo_final['status']=='AUDITORIA_AUTOMATICA_ASTA_CONCLUIDA_AGUARDA_REVISAO_VISUAL'
    assert resumo_final['raw_somente_leitura'] is True
    assert resumo_final['yolo_executado'] is False and resumo_final['hough_executado'] is False
    assert resumo_final['protocolo_tiles_congelado'] is False
    assert resumo_final['revisao_visual_aprovada'] is False
    if problemas:raise RuntimeError(f'Artefatos divergentes: {problemas}')
    print(json.dumps(resumo_final,indent=2,ensure_ascii=False))
    print('BACKUP_AUDITORIA_ASTA_AUTOMATICA_VERIFICADO_SEM_INFERENCIA')

Verificando artefatos ASTA:   0%|          | 0/5 [00:00<?, ?arq/s]

{
  "protocolo": "asta_external_audit_v3",
  "status": "AUDITORIA_AUTOMATICA_ASTA_CONCLUIDA_AGUARDA_REVISAO_VISUAL",
  "record_id": "11642424",
  "doi": "10.5281/zenodo.11642424",
  "pares_auditados": 178,
  "fingerprint_nomes_tamanhos": "6b307e94a0894518016da27c5245b729d113471d6af42393bd6fc167941fb6e0",
  "sha256_contrato_inventario": "ca4609ef651bd30cef611bd394c539868b10f2e8547676e4d00af7e271c14c06",
  "dimensoes_imagem": [
    {
      "largura_imagem": 10560,
      "altura_imagem": 10560,
      "quantidade": 178
    }
  ],
  "modos_imagem": {
    "L": 178
  },
  "valores_mascara": {
    "0|255": 177,
    "0": 1
  },
  "mascaras_vazias": 1,
  "sem_componente_relevante": 1,
  "multiplos_componentes_relevantes": 73,
  "componentes_tocando_borda": 0,
  "area_minima_componente": 15,
  "erros": [],
  "avisos": [
    "1 máscaras vazias exigem revisão.",
    "1 máscaras sem componente >= 15 exigem revisão.",
    "73 máscaras possuem múltiplos componentes relevantes; isso não define sozinho 